Clear permission cache

In [ ]:
from django.core.cache import caches
from django.db import connections
import os
import django
from dotenv import load_dotenv
import sys
import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

# Add the project root to sys.path so 'bcfms' is importable as a package
sys.path.insert(0, "/web_root/bcfms")

# Load environment variables from nr-bcap/.env
load_dotenv(dotenv_path=".env")

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "bcfms.settings")
django.setup()

caches["user_permission"].clear()

with connections["default"].cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM guardian_groupobjectpermission")
    print(cursor.fetchone())

List of user permission cache entries:

In [ ]:
from django.core.cache import caches
from django.conf import settings

def get_all_database_cache_entries(cache_name="default"):
    cache = caches[cache_name]
    cache_table = settings.CACHES[cache_name]["LOCATION"]

    from django.db import connections
    db = getattr(cache, "_db", "default")

    with connections[db].cursor() as cursor:
        cursor.execute(f"SELECT cache_key, value, expires FROM {cache_table}")
        columns = [col[0] for col in cursor.description]
        return [dict(zip(columns, row)) for row in cursor.fetchall()]

entries = get_all_database_cache_entries("user_permission")
for entry in entries:
    print(entry["cache_key"], entry["expires"])

In [ ]:
from django.contrib.auth.models import Group
from django.db import connections

def get_group_permissions(group_name, db="default"):
    # Standard Django group permissions
    group = Group.objects.prefetch_related("permissions__content_type").get(name=group_name)
    django_permissions = group.permissions.all()

    print("=== Django Group Permissions ===")
    for perm in django_permissions:
        print(f"{perm.content_type.app_label}.{perm.codename} — {perm.name}")

    # Guardian object-level permissions
    print("\n=== Guardian Object Permissions ===")
    with connections[db].cursor() as cursor:
        cursor.execute("""
            SELECT
                ggop.id,
                ggop.object_pk,
                ap.codename AS permission_codename,
                ct.app_label,
                ct.model
            FROM guardian_groupobjectpermission ggop
            JOIN auth_permission ap ON ap.id = ggop.permission_id
            JOIN django_content_type ct ON ct.id = ggop.content_type_id
            WHERE ggop.group_id = %s
        """
                       , [group.id])

        columns = [col[0] for col in cursor.description]
        guardian_permissions = [dict(zip(columns, row)) for row in cursor.fetchall()]

    for perm in guardian_permissions:
        print(f"{perm['app_label']}.{perm['permission_codename']} — object_pk: {perm['object_pk']} ({perm['model']})")

    return {
        "django_permissions": django_permissions,
        "guardian_permissions": guardian_permissions,
    }

permissions = get_group_permissions("Guest")

In [ ]:
from django.contrib.auth.models import Group
from guardian.shortcuts import assign_perm,remove_perm
from arches.app.models.models import Node, NodeGroup

def apply_read_nodegroup_permission(node_alias):
    # Get the Guest group
    guest_group = Group.objects.get(name="Guest")

    # Look up the nodegroup via the node alias
    node = Node.objects.select_related("nodegroup").get(alias=node_alias)
    nodegroup = node.nodegroup

    # Assign the read_nodegroup permission
    assign_perm("models.read_nodegroup", guest_group, nodegroup)

    print(f"Assigned 'read_nodegroup' to Guest group for nodegroup: {nodegroup.pk} (via node alias: '{node_alias}')")

def remove_read_nodegroup_permission(node_alias):
    # Get the Guest group
    guest_group = Group.objects.get(name="Guest")

    # Look up the nodegroup via the node alias
    node = Node.objects.select_related("nodegroup").get(alias=node_alias)
    nodegroup = node.nodegroup

    # Assign the read_nodegroup permission
    remove_perm("models.read_nodegroup", guest_group, nodegroup)

    print(f"Removed 'read_nodegroup' to Guest group for nodegroup: {nodegroup.pk} (via node alias: '{node_alias}')")

apply_read_nodegroup_permission("new_node")
# remove_read_nodegroup_permission("new_node")
